# ProRAG: Retrieval Evaluation (v1 vs v2)

This notebook measures **how good the retrieval stage of ProRAG really is**, using 55 hand-checked questions over 3 papers
(*Attention Is All You Need*, the *RAG* paper, *Llama 2*; 111 pages in total).

**What you get**
1. A table comparing 4 retrieval strategies: **vector-only, BM25-only, hybrid (RRF), hybrid + Cohere rerank**.
2. Your **old pipeline (v1) vs the improved one (v2)**.
3. *(Optional)* answer quality: how many answers are correct, and does the system refuse questions that have no answer?

**How to run:** go top to bottom, one cell at a time. Total time is about 25-35 minutes, mostly waiting for Cohere's trial rate limit.
Everything is cached, so if Colab disconnects, run the cells again and it continues where it stopped.

### Words you need to know
- **Gold page**: the page of the PDF that contains the answer (I verified every one against your PDFs).
- **hit@k**: for what % of questions did a chunk from the gold page appear in the top-k results? (hit@3 = 80% means 8 out of 10 questions found the right page in the top 3)
- **MRR** (Mean Reciprocal Rank): rewards putting the right chunk *higher*. Rank 1 scores 1.0, rank 2 scores 0.5, rank 3 scores 0.33, not found scores 0. Average over all questions.
- Questions that compare two papers need *both* papers' gold pages to show up.


## 1. Setup

In [ ]:
!pip -q install chromadb rank-bm25 cohere pypdf langchain-mistralai langchain-groq langchain-text-splitters

In [ ]:
import os
from getpass import getpass

# Paste your keys when asked. They are only kept in this Colab session (never saved in the notebook).
# Press Enter to skip GROQ_API_KEY if you only want Part 1 and 2.
for name in ["MISTRAL_API_KEY", "COHERE_API_KEY", "GROQ_API_KEY"]:
    if name not in os.environ:
        os.environ[name] = getpass(f"Paste {name}: ")
print("keys set:", {k: bool(os.environ[k]) for k in ["MISTRAL_API_KEY", "COHERE_API_KEY", "GROQ_API_KEY"]})

In [ ]:
# Upload attention.pdf, rag.pdf and llama2.pdf (select all three in the file dialog).
need = ["attention.pdf", "rag.pdf", "llama2.pdf"]
if [f for f in need if not os.path.exists(f)]:
    from google.colab import files
    files.upload()
missing = [f for f in need if not os.path.exists(f)]
assert not missing, f"Still missing: {missing}. Names must match exactly (if Colab renamed a file to 'x (1).pdf', rename it)."
print("PDFs ready")

In [ ]:
import re, json, time, pickle, hashlib
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

# ---- Pipeline settings -------------------------------------------------------
CHUNK_SIZE, CHUNK_OVERLAP = 800, 150

# v2 = the improved pipeline you deployed.  v1 = your original pipeline.
V2 = dict(name="v2", sem_k=20, bm25_k=20, fuse_k=15, rerank_k=5, tokenizer="regex")
V1 = dict(name="v1", sem_k=5,  bm25_k=5,  fuse_k=5,  rerank_k=3, tokenizer="split")
#   sem_k / bm25_k : how many candidates each retriever returns
#   fuse_k         : how many RRF-fused candidates go to the reranker
#   rerank_k       : how many chunks survive reranking (= the final context)
#   tokenizer      : "regex" = lowercase + strip punctuation (v2),  "split" = plain .split() (v1)

EVAL_K = 5            # we judge the top-5 results
RERANK_SLEEP = 6.5    # Cohere TRIAL keys allow ~10 rerank calls per minute. Use 0 with a production key.
ANSWER_SLEEP = 10    # Groq FREE tier = only 8,000 tokens/minute for gpt-oss-120b (about 4 answers/minute). Use 0 with a paid key.
JUDGE_SLEEP = 5

def retry(fn, tries=6):
    """Run fn(). If an API error (e.g. rate limit) happens, wait and try again."""
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            if attempt == tries - 1:
                raise
            wait = min(60, 5 * 2 ** attempt)
            print(f"  (API error: {str(e)[:90]} ... retrying in {wait}s)")
            time.sleep(wait)

## 2. Load the PDFs and chunk them (same settings as ProRAG: 800 chars, 150 overlap)

In [ ]:
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

EXPECTED_PAGES = {"attention": 15, "rag": 19, "llama2": 77}
DOCS = list(EXPECTED_PAGES)
page_docs, page_text = [], {}
for d in DOCS:
    reader = PdfReader(f"{d}.pdf")
    n = len(reader.pages)
    print(f"{d:10s} {n} pages", "" if n == EXPECTED_PAGES[d] else f"  WARNING: expected {EXPECTED_PAGES[d]}. Gold pages may not match this file version!")
    for pno, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        page_text[(d, pno)] = text
        page_docs.append(Document(page_content=text, metadata={"source": d, "page": pno}))

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(page_docs)
texts = [c.page_content for c in chunks]
print(f"\n{len(page_docs)} pages -> {len(chunks)} chunks")

## 3. The question set

55 questions: **27 keyword** (exact numbers and names), **13 paraphrase** (the question uses different words than the paper), **10 hard** (tables, several places, or comparing two papers), **5 unanswerable** (topic is close but the answer is NOT in the papers; used to test refusals).
Each question stores the *evidence text* that proves where the answer is, and the gold page(s) were found automatically from that text.

In [ ]:
QUESTIONS = json.loads(r'''[
 {
  "id": "a01",
  "type": "keyword",
  "question": "How many identical layers are stacked in the Transformer encoder?",
  "gold_answer": "N = 6 identical layers (with d_model = 512).",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "The encoder is composed of a stack of N = 6 identical layers"
    ],
    "pages": [
     3
    ]
   }
  ]
 },
 {
  "id": "a02",
  "type": "keyword",
  "question": "What BLEU score did the big Transformer reach on the WMT 2014 English-to-German task?",
  "gold_answer": "28.4 BLEU.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "achieves 28.4 BLEU on the WMT 2014",
     "establishing a new state-of-the-art BLEU score of 28.4"
    ],
    "pages": [
     1,
     8
    ]
   }
  ]
 },
 {
  "id": "a03",
  "type": "keyword",
  "question": "In the original Transformer paper, which beta1, beta2 and epsilon values does the Adam optimizer use?",
  "gold_answer": "beta1 = 0.9, beta2 = 0.98, epsilon = 10^-9.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "We used the Adam optimizer"
    ],
    "pages": [
     7
    ]
   }
  ]
 },
 {
  "id": "a04",
  "type": "keyword",
  "question": "How many warmup steps does the Transformer learning rate schedule use?",
  "gold_answer": "warmup_steps = 4000.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "We used warmup_steps = 4000"
    ],
    "pages": [
     7
    ]
   }
  ]
 },
 {
  "id": "a05",
  "type": "keyword",
  "question": "What is the inner-layer dimensionality of the position-wise feed-forward network in the Transformer?",
  "gold_answer": "d_ff = 2048 (input/output dimension d_model = 512).",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "and the inner-layer has dimensionality"
    ],
    "pages": [
     5
    ]
   }
  ]
 },
 {
  "id": "a06",
  "type": "keyword",
  "question": "How many parallel attention heads does the base Transformer use, and what is the key dimension per head?",
  "gold_answer": "h = 8 heads, with d_k = d_v = 64.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "we employ h = 8 parallel attention layers, or heads"
    ],
    "pages": [
     5
    ]
   }
  ]
 },
 {
  "id": "a07",
  "type": "keyword",
  "question": "In the Transformer paper, what residual dropout rate and label smoothing value were used for the base model?",
  "gold_answer": "Residual dropout P_drop = 0.1 and label smoothing epsilon_ls = 0.1.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "For the base model, we use a rate of",
     "we employed label smoothing of value"
    ],
    "pages": [
     8
    ]
   }
  ]
 },
 {
  "id": "a08",
  "type": "paraphrase",
  "question": "What problem does dividing the attention scores by the square root of the key dimension solve?",
  "gold_answer": "For large d_k the dot products grow large and push the softmax into regions with extremely small gradients; scaling by 1/sqrt(d_k) counteracts this.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "pushing the softmax function into regions where it has"
    ],
    "pages": [
     4
    ]
   }
  ]
 },
 {
  "id": "a09",
  "type": "paraphrase",
  "question": "How does the model know the order of tokens when it has neither recurrence nor convolution?",
  "gold_answer": "It adds positional encodings (sine and cosine functions of different frequencies) to the input embeddings.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "we must inject some information about the relative or absolute position"
    ],
    "pages": [
     6
    ]
   }
  ]
 },
 {
  "id": "a10",
  "type": "paraphrase",
  "question": "What is the benefit of running several attention heads instead of a single one?",
  "gold_answer": "Multi-head attention lets the model jointly attend to information from different representation subspaces at different positions; a single head's averaging inhibits this.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "Multi-head attention allows the model to jointly attend to information from different representation subspaces"
    ],
    "pages": [
     5
    ]
   }
  ]
 },
 {
  "id": "a11",
  "type": "hard",
  "question": "Compare self-attention and recurrent layers in per-layer complexity and number of sequential operations.",
  "gold_answer": "Self-attention: O(n^2 * d) per layer and O(1) sequential operations. Recurrent: O(n * d^2) per layer and O(n) sequential operations.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "a self-attention layer connects all positions with a constant number of sequentially",
     "Maximum path lengths, per-layer complexity and minimum number of sequential operations"
    ],
    "pages": [
     6
    ]
   }
  ]
 },
 {
  "id": "a12",
  "type": "hard",
  "question": "How long were the base and big Transformer models trained, and on what hardware?",
  "gold_answer": "Base: 100,000 steps (about 12 hours); big: 300,000 steps (3.5 days); on one machine with 8 NVIDIA P100 GPUs.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "We trained the base models for a total of 100,000 steps or 12 hours"
    ],
    "pages": [
     7
    ]
   }
  ]
 },
 {
  "id": "a13",
  "type": "hard",
  "question": "What F1 did the Transformer reach on WSJ constituency parsing, trained on WSJ only and semi-supervised?",
  "gold_answer": "91.3 F1 (WSJ only) and 92.7 F1 (semi-supervised).",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "The Transformer generalizes well to English constituency parsing",
     "Transformer (4 layers) semi-supervised 92.7"
    ],
    "pages": [
     10
    ]
   }
  ]
 },
 {
  "id": "a14",
  "type": "keyword",
  "question": "How large was the WMT 2014 English-German training set and what vocabulary did the Transformer use for it?",
  "gold_answer": "About 4.5 million sentence pairs, with a shared byte-pair-encoding vocabulary of about 37,000 tokens.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "consisting of about 4.5 million"
    ],
    "pages": [
     7
    ]
   }
  ]
 },
 {
  "id": "r01",
  "type": "keyword",
  "question": "How many trainable parameters do the RAG models contain in total?",
  "gold_answer": "About 626M trainable parameters (BERT-base encoders of 110M each, plus 406M from BART-large).",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "making a total of 626M trainable"
    ],
    "pages": [
     18
    ]
   }
  ]
 },
 {
  "id": "r02",
  "type": "keyword",
  "question": "How is the Wikipedia knowledge source split into documents for RAG, and how many documents result?",
  "gold_answer": "Each article of the December 2018 dump is split into disjoint 100-word chunks, giving 21M documents.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "Each Wikipedia article is split into disjoint 100-word chunks"
    ],
    "pages": [
     4
    ]
   }
  ]
 },
 {
  "id": "r03",
  "type": "keyword",
  "question": "What Exact Match did RAG-Sequence score on the Natural Questions test set?",
  "gold_answer": "44.5 EM.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "RAG-Seq. 44.5 56.8/68.0 45.2 52.2"
    ],
    "pages": [
     6
    ]
   }
  ]
 },
 {
  "id": "r04",
  "type": "keyword",
  "question": "In the index hot-swapping experiment, how accurate was RAG when the index year matched the world leaders' year?",
  "gold_answer": "70% with the 2016 index for 2016 leaders and 68% with the 2018 index for 2018 leaders (mismatched: 12% and 4%).",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "RAG answers 70% correctly using the 2016 index for"
    ],
    "pages": [
     8
    ]
   }
  ]
 },
 {
  "id": "r05",
  "type": "keyword",
  "question": "In the Jeopardy human evaluation, in what percentage of cases was RAG judged more factual than BART?",
  "gold_answer": "42.7% (BART was more factual in only 7.1%).",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "while RAG was more factual in 42.7% of cases"
    ],
    "pages": [
     6
    ]
   }
  ]
 },
 {
  "id": "r06",
  "type": "keyword",
  "question": "Which retriever does RAG use and what encoders does it have?",
  "gold_answer": "A DPR bi-encoder with a BERT-base query encoder and a BERT-base document encoder, searched with Maximum Inner Product Search.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "is a dense representation of a document produced by a BERTBASE document encoder"
    ],
    "pages": [
     3
    ]
   }
  ]
 },
 {
  "id": "r07",
  "type": "paraphrase",
  "question": "How does the RAG-Sequence model differ from RAG-Token in how it uses retrieved passages?",
  "gold_answer": "RAG-Sequence uses the same retrieved document for the whole output sequence; RAG-Token can use a different document for each output token.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "In one approach, RAG-Sequence, the model uses the same document to predict each target token"
    ],
    "pages": [
     3
    ]
   }
  ]
 },
 {
  "id": "r08",
  "type": "paraphrase",
  "question": "Why do the authors not update the document encoder during RAG training?",
  "gold_answer": "Updating it is costly because the document index would have to be rebuilt periodically; they found it unnecessary and fine-tune only the query encoder and the generator.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "Updating the document encoder BERTd during training is costly"
    ],
    "pages": [
     4
    ]
   }
  ]
 },
 {
  "id": "r09",
  "type": "paraphrase",
  "question": "How can the knowledge of a RAG model be refreshed without retraining it?",
  "gold_answer": "By replacing its non-parametric memory, the document index (index hot-swapping).",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "we can update RAG's world knowledge by simply replacing its non-parametric memory",
     "Parametric-only models like T5 or BART need further training"
    ],
    "pages": [
     7,
     8
    ]
   }
  ]
 },
 {
  "id": "r10",
  "type": "paraphrase",
  "question": "Why did a keyword-based retriever beat the learned retriever on the fact verification task?",
  "gold_answer": "FEVER claims are heavily entity-centric and therefore well suited to word-overlap retrieval such as BM25.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "FEVER claims are heavily entity-centric and thus well-suited for word overlap-based retrieval"
    ],
    "pages": [
     7
    ]
   }
  ]
 },
 {
  "id": "r11",
  "type": "paraphrase",
  "question": "What is retrieval collapse in RAG training?",
  "gold_answer": "The retriever learns to return the same documents regardless of the input, so the generator learns to ignore them (observed e.g. for story generation).",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "would collapse and learn to retrieve the same documents regardless of the input"
    ],
    "pages": [
     19
    ]
   }
  ]
 },
 {
  "id": "r12",
  "type": "hard",
  "question": "How many documents does RAG retrieve at test time for open-domain QA with RAG-Token versus RAG-Sequence?",
  "gold_answer": "15 documents for RAG-Token and 50 for RAG-Sequence (using Thorough Decoding); 10 for MS-MARCO and Jeopardy.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "using 15 retrieved documents for RAG-Token models"
    ],
    "pages": [
     17
    ]
   }
  ]
 },
 {
  "id": "r13",
  "type": "hard",
  "question": "What are the train, development and test sizes of the Natural Questions dataset used for RAG?",
  "gold_answer": "Train 79,169; development 8,758; test 3,611.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "Natural Questions 79169 8758 3611"
    ],
    "pages": [
     19
    ]
   }
  ]
 },
 {
  "id": "r14",
  "type": "hard",
  "question": "How did RAG deal with CuratedTrec answers given as regular expressions?",
  "gold_answer": "It retrieved the top 1000 documents per query and used the answer that most frequently matches the regex pattern as the supervision target.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "use the answer that most frequently matches the regex pattern as the supervision target"
    ],
    "pages": [
     18
    ]
   }
  ]
 },
 {
  "id": "r15",
  "type": "keyword",
  "question": "Which three class labels does the FEVER task use for claim classification?",
  "gold_answer": "Supports, refutes, and not enough info.",
  "groups": [
   {
    "doc": "rag",
    "evidence": [
     "We map FEVER class labels (supports, refutes, or not enough info) to single output tokens"
    ],
    "pages": [
     5
    ]
   }
  ]
 },
 {
  "id": "l01",
  "type": "keyword",
  "question": "Which AdamW hyperparameters were used to pretrain Llama 2?",
  "gold_answer": "AdamW with beta1 = 0.9, beta2 = 0.95, eps = 1e-5; cosine schedule with 2000 warmup steps, final LR 10% of peak; weight decay 0.1; gradient clipping 1.0.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "We trained using the AdamW optimizer"
    ],
    "pages": [
     5
    ]
   }
  ]
 },
 {
  "id": "l02",
  "type": "keyword",
  "question": "On how many tokens was Llama 2 pretrained?",
  "gold_answer": "2 trillion tokens (context length 4k).",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "We trained on 2 trillion tokens of data"
    ],
    "pages": [
     5
    ]
   }
  ]
 },
 {
  "id": "l03",
  "type": "keyword",
  "question": "What is the vocabulary size of the Llama 2 tokenizer?",
  "gold_answer": "32k tokens (BPE via SentencePiece).",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "The total vocabulary size is 32k tokens"
    ],
    "pages": [
     6
    ]
   }
  ]
 },
 {
  "id": "l04",
  "type": "keyword",
  "question": "How many GPU hours were needed to pretrain Llama 2 70B and how much carbon did it emit?",
  "gold_answer": "1,720,320 GPU hours and 291.42 tCO2eq.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "70B 1720320 400 291.42"
    ],
    "pages": [
     7
    ]
   }
  ]
 },
 {
  "id": "l05",
  "type": "keyword",
  "question": "How many annotations did the Llama 2 team collect for supervised fine-tuning before they stopped?",
  "gold_answer": "27,540 annotations.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "We stopped annotating SFT after collecting a total of 27,540 annotations"
    ],
    "pages": [
     9
    ]
   }
  ]
 },
 {
  "id": "l06",
  "type": "keyword",
  "question": "What learning rate, batch size, sequence length and number of epochs were used for Llama 2 supervised fine-tuning?",
  "gold_answer": "Cosine schedule starting at 2e-5, weight decay 0.1, batch size 64, sequence length 4096 tokens, 2 epochs.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "a weight decay of 0.1, a batch size of 64, and a sequence length of 4096 tokens",
     "we fine-tune the model for 2 epochs"
    ],
    "pages": [
     9
    ]
   }
  ]
 },
 {
  "id": "l07",
  "type": "keyword",
  "question": "What batch size and clip threshold were used for each PPO iteration in Llama 2-Chat training?",
  "gold_answer": "Batch size 512, PPO clip threshold 0.2, mini-batch size 64, one gradient step per mini-batch.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "a PPO clip threshold of0.2"
    ],
    "pages": [
     15
    ]
   }
  ]
 },
 {
  "id": "l08",
  "type": "keyword",
  "question": "What share of the Llama 2 safety preference data has both responses safe?",
  "gold_answer": "47% (18% preferred response safe and other unsafe; 35% both unsafe).",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "with 18%, 47%, and 35% of the safety dataset falling into each bin"
    ],
    "pages": [
     10
    ]
   }
  ]
 },
 {
  "id": "l09",
  "type": "keyword",
  "question": "How many comparisons does the Meta Safety and Helpfulness preference dataset contain?",
  "gold_answer": "1,418,091 comparisons.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "Meta (Safety & Helpfulness) 1,418,091"
    ],
    "pages": [
     11
    ]
   }
  ]
 },
 {
  "id": "l10",
  "type": "keyword",
  "question": "For how many epochs were the Llama 2 reward models trained, and what maximum learning rates were used?",
  "gold_answer": "One epoch; maximum learning rate 5e-6 for the 70B model and 1e-5 for the others.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "We train for one epoch over the training data"
    ],
    "pages": [
     12
    ]
   }
  ]
 },
 {
  "id": "l11",
  "type": "paraphrase",
  "question": "Which technique helps Llama 2-Chat keep following an instruction over many dialogue turns?",
  "gold_answer": "Ghost Attention (GAtt).",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "we propose Ghost Attention (GAtt)"
    ],
    "pages": [
     16
    ]
   }
  ]
 },
 {
  "id": "l12",
  "type": "paraphrase",
  "question": "How does safety context distillation work in Llama 2-Chat?",
  "gold_answer": "Prompts are prefixed with a safety preprompt to generate safer responses, then the model is fine-tuned on those safer responses without the preprompt.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "generating safer model responses by prefixing a prompt with a safety preprompt"
    ],
    "pages": [
     23
    ]
   }
  ]
 },
 {
  "id": "l13",
  "type": "paraphrase",
  "question": "Why did the Llama 2 team prefer a small set of own annotations over millions of third-party fine-tuning examples?",
  "gold_answer": "Third-party data lacked diversity and quality; fewer but higher-quality vendor annotations notably improved results.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "By setting aside millions of examples from third-party datasets"
    ],
    "pages": [
     9
    ]
   }
  ]
 },
 {
  "id": "l14",
  "type": "paraphrase",
  "question": "Which attention variant do the larger Llama 2 models use to make inference more scalable?",
  "gold_answer": "Grouped-query attention (GQA), in the 34B and 70B models.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "Bigger models 34B and 70B use Grouped-Query Attention (GQA) for improved inference scalability"
    ],
    "pages": [
     6
    ]
   }
  ]
 },
 {
  "id": "l15",
  "type": "paraphrase",
  "question": "Which two reinforcement learning methods were used to refine Llama 2-Chat?",
  "gold_answer": "Rejection sampling fine-tuning and Proximal Policy Optimization (PPO).",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "In Rejection Sampling fine-tuning, we sample all the outputs given the initial policy",
     "specifically through rejection sampling and Proximal Policy"
    ],
    "pages": [
     5,
     14
    ]
   }
  ]
 },
 {
  "id": "l16",
  "type": "hard",
  "question": "What truthfulness and toxicity scores did the pretrained Llama 2 70B get on TruthfulQA and ToxiGen?",
  "gold_answer": "TruthfulQA 50.18 (higher is better) and ToxiGen 24.60 (lower is better).",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "34B 43.45 21.19 70B 50.18 24.60"
    ],
    "pages": [
     23
    ]
   }
  ]
 },
 {
  "id": "l17",
  "type": "hard",
  "question": "How does the Llama 2 reward model loss take into account how much better the chosen response is?",
  "gold_answer": "A margin term m(r) based on the preference rating is added to the binary ranking loss; larger margins for clearly distinct responses.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "we further add a margin component in the loss"
    ],
    "pages": [
     11
    ]
   }
  ]
 },
 {
  "id": "l18",
  "type": "keyword",
  "question": "How many GPU hours in total did Llama 2 pretraining take, and how much was emitted overall?",
  "gold_answer": "3.3M GPU hours on A100-80GB; 539 tCO2eq, 100% offset by Meta's sustainability program.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "A cumulative of 3.3M GPU hours of computation was performed"
    ],
    "pages": [
     7
    ]
   }
  ]
 },
 {
  "id": "l19",
  "type": "keyword",
  "question": "On what clusters and GPUs was Llama 2 pretrained?",
  "gold_answer": "Meta's Research SuperCluster (RSC) and internal production clusters, both with NVIDIA A100 GPUs.",
  "groups": [
   {
    "doc": "llama2",
    "evidence": [
     "pretrained our models on Meta's Research SuperCluster"
    ],
    "pages": [
     6
    ]
   }
  ]
 },
 {
  "id": "m01",
  "type": "hard",
  "question": "Compare the number of learning-rate warmup steps of the original Transformer and of Llama 2 pretraining.",
  "gold_answer": "Transformer: 4000 warmup steps. Llama 2: 2000 warmup steps.",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "We used warmup_steps = 4000"
    ],
    "pages": [
     7
    ]
   },
   {
    "doc": "llama2",
    "evidence": [
     "with warmup of 2000 steps"
    ],
    "pages": [
     5
    ]
   }
  ]
 },
 {
  "id": "m02",
  "type": "hard",
  "question": "How do the Transformer and Llama 2 differ in how they encode token positions?",
  "gold_answer": "Transformer: fixed sinusoidal (sine/cosine) positional encodings. Llama 2: rotary positional embeddings (RoPE).",
  "groups": [
   {
    "doc": "attention",
    "evidence": [
     "we use sine and cosine functions of different frequencies"
    ],
    "pages": [
     6
    ]
   },
   {
    "doc": "llama2",
    "evidence": [
     "rotary positional embeddings"
    ],
    "pages": [
     5
    ]
   }
  ]
 },
 {
  "id": "u01",
  "type": "unanswerable",
  "question": "What BLEU score did the Transformer achieve on the WMT 2014 English-to-Czech translation task?",
  "gold_answer": "Not in the documents.",
  "groups": []
 },
 {
  "id": "u02",
  "type": "unanswerable",
  "question": "What score did the RAG model achieve on the HotpotQA dataset?",
  "gold_answer": "Not in the documents.",
  "groups": []
 },
 {
  "id": "u03",
  "type": "unanswerable",
  "question": "What is the score of Llama 2-Chat on the MMLU-Pro benchmark?",
  "gold_answer": "Not in the documents.",
  "groups": []
 },
 {
  "id": "u04",
  "type": "unanswerable",
  "question": "Who is the CEO of Meta that announced the release of Llama 2?",
  "gold_answer": "Not in the documents.",
  "groups": []
 },
 {
  "id": "u05",
  "type": "unanswerable",
  "question": "What was the total monetary cost of training the big Transformer model?",
  "gold_answer": "Not in the documents.",
  "groups": []
 }
]''')
answerable = [q for q in QUESTIONS if q['type'] != 'unanswerable']
print(len(QUESTIONS), 'questions;', len(answerable), 'answerable')
print(pd.Series([q['type'] for q in QUESTIONS]).value_counts().to_string())

In [ ]:
# Sanity check: re-find every evidence snippet in YOUR PDFs and confirm the gold pages.
def norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())   # ignore spaces/punctuation (some PDFs lose spaces when extracted)

page_norm = {k: norm(v) for k, v in page_text.items()}
problems = []
for q in QUESTIONS:
    for g in q["groups"]:
        found = sorted({p for (d, p), t in page_norm.items() if d == g["doc"] and any(norm(ev) in t for ev in g["evidence"])})
        if found != g["pages"]:
            problems.append((q["id"], "expected", g["pages"], "found", found))
print("All gold pages verified against your PDFs" if not problems else f"PROBLEMS: {problems}")

## 4. Embed all chunks and build the search indexes
This calls the Mistral embedding API (about 20 calls). The vectors are cached in `emb_cache.pkl`, so this only happens once.

In [ ]:
import chromadb
from langchain_mistralai import MistralAIEmbeddings
from rank_bm25 import BM25Okapi

emb = MistralAIEmbeddings(model="mistral-embed")

sig = hashlib.md5("\n".join(texts).encode()).hexdigest()
vectors = None
if os.path.exists("emb_cache.pkl"):
    saved = pickle.load(open("emb_cache.pkl", "rb"))
    if saved["sig"] == sig:
        vectors = saved["vectors"]
        print("loaded cached embeddings")
if vectors is None:
    vectors, B = [], 32
    for i in range(0, len(texts), B):
        batch = texts[i:i + B]
        vectors.extend(retry(lambda: emb.embed_documents(batch)))
        print(f"embedded {min(i + B, len(texts))}/{len(texts)}", end="\r")
        time.sleep(0.5)
    pickle.dump({"sig": sig, "vectors": vectors}, open("emb_cache.pkl", "wb"))

# Vector index (ChromaDB, in memory, same default settings as ProRAG)
client = chromadb.EphemeralClient()
try:
    client.delete_collection("eval")
except Exception:
    pass
col = client.create_collection("eval")
for i in range(0, len(chunks), 256):
    j = min(i + 256, len(chunks))
    col.add(ids=[str(x) for x in range(i, j)], embeddings=vectors[i:j], documents=texts[i:j], metadatas=[c.metadata for c in chunks[i:j]])

# Keyword indexes (BM25): v2 tokenizer (lowercase, no punctuation) and v1 tokenizer (plain split)
def tok_regex(t): return re.findall(r"\w+", t.lower())
bm25_v2 = BM25Okapi([tok_regex(t) for t in texts])
bm25_v1 = BM25Okapi([t.split() for t in texts])
print(f"\nindexes ready: {col.count()} vectors, {len(texts)} BM25 documents")

## 5. The retrieval pipeline
Same logic as `retriever.py` in ProRAG. For a fair test every strategy gets the **raw question** (no query rewriting; that is a separate experiment).

In [ ]:
import cohere
co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"])

qvec_cache = {}
def embed_query(q):
    if q not in qvec_cache:
        qvec_cache[q] = retry(lambda: emb.embed_query(q))
    return qvec_cache[q]

def vector_search(q, k):
    res = col.query(query_embeddings=[embed_query(q)], n_results=k)
    return [int(i) for i in res["ids"][0]]

def bm25_search(q, k, cfg):
    if cfg["tokenizer"] == "regex":
        toks = tok_regex(q); scores = bm25_v2.get_scores(toks); drop_zero = True
    else:
        toks = q.split();    scores = bm25_v1.get_scores(toks); drop_zero = False
    if not toks:
        return []
    order = np.argsort(-scores, kind="stable")[:k]
    return [int(i) for i in order if (scores[i] > 0 or not drop_zero)]

def rrf(rankings, k=60):
    """Reciprocal Rank Fusion: score = sum of 1/(k+rank) over every list the chunk appears in."""
    scores = {}
    for ranking in rankings:
        for rank, i in enumerate(ranking, start=1):
            scores[i] = scores.get(i, 0) + 1 / (k + rank)
    return [i for i, _ in sorted(scores.items(), key=lambda p: p[1], reverse=True)]

# Rerank results are cached on disk so a disconnect never makes you repeat paid/rate-limited calls.
RERANK_FILE = "rerank_cache.json"
rerank_cache = json.load(open(RERANK_FILE)) if os.path.exists(RERANK_FILE) else {}
rerank_calls = {"n": 0}

def rerank(q, cand_ids, top_n, tag):
    key = tag + "|" + hashlib.md5((q + ",".join(map(str, cand_ids))).encode()).hexdigest()
    if key not in rerank_cache:
        resp = retry(lambda: co.rerank(model="rerank-v3.5", query=q, documents=[texts[i] for i in cand_ids], top_n=top_n))
        rerank_cache[key] = [[cand_ids[r.index], float(r.relevance_score)] for r in resp.results]
        json.dump(rerank_cache, open(RERANK_FILE, "w"))
        rerank_calls["n"] += 1
        time.sleep(RERANK_SLEEP)
    return rerank_cache[key]

def run_pipeline(q, mode, cfg):
    """Returns the ranked list of chunk ids for one question. mode: vector | bm25 | hybrid | hybrid_rerank"""
    sem = vector_search(q, cfg["sem_k"])
    if mode == "vector":
        return sem[:EVAL_K]
    kw = bm25_search(q, cfg["bm25_k"], cfg)
    if mode == "bm25":
        return kw[:EVAL_K]
    fused = rrf([sem, kw])
    if mode == "hybrid":
        return fused[:EVAL_K]
    ranked = rerank(q, fused[:cfg["fuse_k"]], cfg["rerank_k"], tag=cfg["name"])
    return [i for i, _ in ranked]

print("pipeline ready")

## 6. Metrics

In [ ]:
def first_hit(ranked, q, K):
    """Smallest k (<= K) such that, within the top-k, EVERY gold group has a chunk from one of its gold pages."""
    for k in range(1, K + 1):
        top = ranked[:k]
        if all(any(chunks[i].metadata["source"] == g["doc"] and chunks[i].metadata["page"] in g["pages"] for i in top)
               for g in q["groups"]):
            return k
    return None

def evaluate(rank_lists, qs, K=5):
    rows = []
    for q in qs:
        fh = first_hit(rank_lists[q["id"]][:K], q, K)
        row = {"id": q["id"], "type": q["type"],
               "doc": q["groups"][0]["doc"] if len(q["groups"]) == 1 else "multi",
               "first_hit_rank": fh}
        for k in (1, 3, 5):
            if k <= K:
                row[f"hit@{k}"] = int(fh is not None and fh <= k)
        row[f"MRR@{K}"] = 1 / fh if fh else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

def summarize(df, K=5):
    out = {c: round(df[c].mean() * 100, 1) for c in df.columns if c.startswith("hit@")}   # percentages
    out[f"MRR@{K}"] = round(df[f"MRR@{K}"].mean(), 3)                                      # 0..1
    return out

## 7. PART 1: compare the 4 retrieval strategies (v2 settings)
The first three are quick. `hybrid_rerank` makes one Cohere call per question (~50 calls, about 6 minutes on a trial key).

In [ ]:
MODES = ["vector", "bm25", "hybrid", "hybrid_rerank"]
results = {}
for mode in MODES:
    t0 = time.time(); out = {}
    for n, q in enumerate(answerable, start=1):
        out[q["id"]] = run_pipeline(q["question"], mode, V2)
        if mode == "hybrid_rerank" and n % 10 == 0:
            print(f"  hybrid_rerank: {n}/{len(answerable)} questions")
    results[mode] = out
    print(f"{mode:14s} done in {time.time() - t0:.0f}s")

In [ ]:
part1 = pd.DataFrame({m: summarize(evaluate(results[m], answerable, K=5), K=5) for m in MODES}).T
part1.index.name = "strategy"
print("PART 1: retrieval quality on", len(answerable), "answerable questions (hit@k in %, MRR 0..1)")
display(part1)

def breakdown(by, metric="hit@3"):
    return pd.DataFrame({m: (evaluate(results[m], answerable, K=5).groupby(by)[metric].mean() * 100).round(1) for m in MODES})

print("\nhit@3 (%) by question type:");  display(breakdown("type"))
print("hit@3 (%) by document:");         display(breakdown("doc"))

**How to read this:**
- If *keyword* questions favour BM25 and *paraphrase* questions favour vector search, hybrid exists for a reason.
- The *llama2* column is the interesting one: that PDF extracts with many words stuck together (e.g. `Tocreatethenewfamilyof`). Words that are glued together can't match a keyword search, so check whether BM25 does worse there.
- Does reranking raise hit@1 (the best chunk comes first)? That is what a reranker is for.

## 8. PART 2: your old pipeline (v1) vs the improved one (v2)
v1 = plain `.split()` BM25, top-5 + top-5, rerank down to 3. v2 = regex tokenizer, top-20 + top-20, 15 to the reranker, keep 5.
Both are scored on their **top-3** so the comparison is fair. This runs another ~50 Cohere calls for v1 (about 6 minutes).

In [ ]:
rows = []
def add(label, lists):
    rows.append({"setup": label, **summarize(evaluate(lists, answerable, K=3), K=3)})

for mode, label in [("bm25", "BM25 only"), ("hybrid", "Hybrid (RRF)"), ("hybrid_rerank", "Hybrid + rerank")]:
    lists_v1 = {q["id"]: run_pipeline(q["question"], mode, V1) for q in answerable}
    add(f"{label} | v1", lists_v1)
    add(f"{label} | v2", results[mode])
part2 = pd.DataFrame(rows).set_index("setup")
print("PART 2: v1 vs v2, scored on top-3 (hit in %, MRR 0..1)")
display(part2)

## 9. Look at the failures (this is where you learn the most)

In [ ]:
df_best = evaluate(results["hybrid_rerank"], answerable, K=5)
missed = df_best[df_best["hit@5"] == 0]["id"].tolist()
qmap = {q["id"]: q for q in QUESTIONS}
print(f"hybrid_rerank missed {len(missed)} of {len(answerable)} questions in the top-5\n")
for qid in missed:
    q = qmap[qid]
    print(f"[{qid}] ({q['type']}) {q['question']}")
    print("   gold     :", [(g["doc"], g["pages"]) for g in q["groups"]])
    print("   retrieved:", [(chunks[i].metadata["source"], chunks[i].metadata["page"]) for i in results["hybrid_rerank"][qid]])
    print()

## 10. PART 3 (optional): answer quality
Needs a Groq key. For each question: retrieve with the final pipeline (v2 hybrid + rerank), generate an answer with the same prompt as ProRAG, then
- **answerable**: an LLM judge compares the answer with the reference answer (CORRECT / INCORRECT);
- **unanswerable**: did the system correctly say *"I could not find the answer in the provided documents"*?

**Time and limits:** Groq's free tier allows only 8,000 tokens/minute and 200,000 tokens/day for `gpt-oss-120b`, so this part takes about **20-25 minutes**.
Answers use `gpt-oss-120b` (the model in your app); grading uses the smaller `gpt-oss-20b`, which has its own separate quota.
Progress is saved in `part3_cache.json`: if you hit a limit or Colab disconnects, just run this cell again (or tomorrow). Parts 1 and 2 do not depend on it.

Caveat: an LLM judge is useful but imperfect. Read some answers yourself (saved in `part3_answers.csv`).

In [ ]:
from langchain_groq import ChatGroq

ANSWER_MODEL = "openai/gpt-oss-120b"   # same model as the ProRAG app
JUDGE_MODEL = "openai/gpt-oss-20b"     # smaller model used only for grading (separate free-tier quota)
NOT_FOUND = "I could not find the answer in the provided documents."
P3_FILE = "part3_cache.json"           # delete this file to start Part 3 from scratch

def build_context(ids):
    return "\n\n".join(f"[{n}] ({chunks[i].metadata['source']}, page {chunks[i].metadata['page']})\n{chunks[i].page_content}"
                        for n, i in enumerate(ids, start=1))

def answer_prompt(question, ids):
    return f"""
You are a helpful research assistant.

Answer the user's question using ONLY the provided context.

Rules:
- Do not use outside knowledge.
- Do not make up information.
- Give one clear and concise answer.
- Cite the supporting context chunks inline using their numbers, for example [1] or [2][3]. Only cite chunks you actually used.
- If the context does not contain enough information, reply with exactly:
  "{NOT_FOUND}"

User Question:
{question}

Context:
{build_context(ids)}
"""

def judge_prompt(question, gold, pred):
    return f"""You are grading an answer to a question about a research paper.
Question: {question}
Reference answer: {gold}
Model answer: {pred}
Reply CORRECT if the model answer contains the key facts/numbers of the reference answer (extra detail is fine, contradictions are not).
Otherwise reply INCORRECT. Reply with one word only."""

def run_part3():
    llm_answer = ChatGroq(model=ANSWER_MODEL, temperature=0)
    llm_judge = ChatGroq(model=JUDGE_MODEL, temperature=0)
    cache = json.load(open(P3_FILE)) if os.path.exists(P3_FILE) else {}
    rows = []
    for n, q in enumerate(QUESTIONS, start=1):
        c = cache.setdefault(q["id"], {})
        ids = results["hybrid_rerank"].get(q["id"]) or run_pipeline(q["question"], "hybrid_rerank", V2)
        if "answer" not in c:
            c["answer"] = retry(lambda: llm_answer.invoke(answer_prompt(q["question"], ids)).content)
            json.dump(cache, open(P3_FILE, "w")); time.sleep(ANSWER_SLEEP)
        ans = c["answer"]
        refused = "could not find the answer" in ans.lower()
        row = {"id": q["id"], "type": q["type"], "question": q["question"], "answer": ans, "refused": refused}
        if q["type"] == "unanswerable":
            row["correct"] = refused                      # the right behaviour is to refuse
        else:
            row["retrieval_hit@5"] = int(first_hit(ids, q, EVAL_K) is not None)
            if refused:
                row["correct"] = False
            else:
                if "judge" not in c:
                    out = retry(lambda: llm_judge.invoke(judge_prompt(q["question"], q["gold_answer"], ans)).content).upper()
                    c["judge"] = ("INCORRECT" not in out) and ("CORRECT" in out)
                    json.dump(cache, open(P3_FILE, "w")); time.sleep(JUDGE_SLEEP)
                row["correct"] = c["judge"]
        rows.append(row)
        if n % 10 == 0:
            print(f"  done {n}/{len(QUESTIONS)}")
    return pd.DataFrame(rows)

df3 = run_part3() if os.environ.get("GROQ_API_KEY") else None
if df3 is None:
    print("No GROQ_API_KEY set: Part 3 skipped (Parts 1 and 2 are complete).")
else:
    ans_df = df3[df3["type"] != "unanswerable"].copy()
    una_df = df3[df3["type"] == "unanswerable"]
    ans_df["retrieval_hit@5"] = ans_df["retrieval_hit@5"].astype(int)
    ans_df["correct"] = ans_df["correct"].astype(bool)
    print(f"\nCorrect answers (answerable)   : {ans_df['correct'].sum()}/{len(ans_df)}  ({ans_df['correct'].mean()*100:.1f}%)")
    print(f"Correct refusals (unanswerable): {una_df['correct'].sum()}/{len(una_df)}")
    print("\nWere failures caused by retrieval or by generation?  (rows: gold page retrieved? / columns: answer correct?)")
    display(pd.crosstab(ans_df["retrieval_hit@5"], ans_df["correct"]))
    print("Correctness by question type (%):")
    display((ans_df.groupby("type")["correct"].mean() * 100).round(1))

## 11. Save everything

In [ ]:
part1.to_csv("part1_strategies.csv")
part2.to_csv("part2_v1_vs_v2.csv")
per_q = pd.DataFrame({"id": [q["id"] for q in answerable], "type": [q["type"] for q in answerable], "question": [q["question"] for q in answerable]})
for m in MODES:
    per_q[f"{m}_first_hit_rank"] = evaluate(results[m], answerable, K=5)["first_hit_rank"].values
per_q.to_csv("per_question_results.csv", index=False)
if df3 is not None:
    df3.to_csv("part3_answers.csv", index=False)

try:
    from google.colab import files
    for f in ["part1_strategies.csv", "part2_v1_vs_v2.csv", "per_question_results.csv", "part3_answers.csv"]:
        if os.path.exists(f):
            files.download(f)
except ImportError:
    print("saved CSV files in the current folder")
print("\nPART 1"); display(part1)
print("PART 2"); display(part2)

## What can go on your CV?

Only numbers **printed by this notebook, from a run you did yourself**, and only claims you can explain in an interview. For example:

> *Built a 55-question retrieval benchmark over 3 papers (111 pages) with page-level ground truth; compared vector, BM25, hybrid (RRF) and hybrid + Cohere rerank. Hybrid + rerank reached hit@3 of **X%** (vs **Y%** for vector-only); the v2 pipeline improved hit@3 from **A%** to **B%** over my v1 baseline.*

If a result is unflattering (say, reranking doesn't help on this dataset), **report it honestly and explain why**. That shows more engineering maturity than a perfect-looking number.

**Next experiments:** query rewriting on vs off, chunk size 400 / 800 / 1200, a minimum rerank-score cutoff, a bigger question set.